# Phase 5 — Hyperparameter Tuning

## Objective

The objective of this phase is to improve the performance of the best-performing baseline models from Phase 4 using hyperparameter tuning.

Since the business objective prioritizes identifying churned customers, the primary optimization metric will be:

**Recall**

This is because missing an actual churned customer is more costly than incorrectly flagging an existing customer as likely to churn.

## Models Selected for Tuning

Based on Phase 4 results, only the following models will be tuned:

1. Random Forest
2. XGBoost

## Reason

Random Forest currently has the best recall, while XGBoost has the best overall performance across accuracy, precision, F1 score, and ROC-AUC.

## Important Rule

No raw preprocessing will be repeated in this phase.

The already processed datasets from `data/processed/` will be reused directly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Model selection
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## 5.1 Why Default Parameters Are Not Enough

Default hyperparameters are general-purpose settings provided by machine learning libraries. These values are not optimized for a specific dataset, feature distribution, class imbalance, or business objective.

In this project, the primary objective is to identify customers who are likely to churn. Therefore, recall is prioritized because false negatives represent churned customers that the bank fails to identify.

Hyperparameter tuning is required to search for model configurations that improve recall while maintaining acceptable precision, F1 score, and ROC-AUC.

## 5.2 GridSearchCV

GridSearchCV is a hyperparameter tuning technique that exhaustively searches through all possible combinations of the specified hyperparameter values. Each combination is evaluated using cross-validation.

It is useful when the search space is small and controlled. However, it can become computationally expensive when many parameters or values are included.

In this project, GridSearchCV will be considered only for compact search spaces, while larger search spaces will be handled using RandomizedSearchCV.

## 5.3 RandomizedSearchCV

RandomizedSearchCV is a hyperparameter optimization technique that evaluates a fixed number of randomly selected hyperparameter combinations from a predefined search space.

Unlike GridSearchCV, which exhaustively evaluates all combinations, RandomizedSearchCV provides a computationally efficient alternative, especially for models with many influential hyperparameters.

In this project, RandomizedSearchCV was preferred for XGBoost due to its large hyperparameter space. The tuning objective remained aligned with the business requirement of maximizing recall to minimize missed churners.

## 5.4 Random Forest Hyperparameter Tuning Strategy

RandomizedSearchCV was selected to tune the Random Forest classifier due to its computational efficiency and practical relevance in real-world applications.

Only the most influential hyperparameters were considered:

- n_estimators
- max_depth
- min_samples_split
- min_samples_leaf
- max_features

Stratified 5-fold cross-validation was used to preserve the class distribution across folds.

Recall was chosen as the optimization metric because the business objective prioritized minimizing false negatives, thereby reducing the likelihood of failing to identify customers at risk of churning.

In [2]:
# Load processed datasets
X_train = pd.read_csv("../data/processed/X_train_processed.csv")
X_test = pd.read_csv("../data/processed/X_test_processed.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8101, 37)
X_test shape: (2026, 37)
y_train shape: (8101,)
y_test shape: (2026,)


In [3]:
# Cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [5]:
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="recall",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [6]:
rf_random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END max_depth=40, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time=   2.2s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time=   2.3s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time=   2.3s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time=   2.3s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time=   2.3s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=500; total time=   3.4s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=500; total time=   3.7s
[CV] END max_depth=40, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=50

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` 

In [7]:
print("Best Random Forest Parameters:")
print(rf_random_search.best_params_)

print("\nBest Cross-Validation Recall:")
print(rf_random_search.best_score_)

Best Random Forest Parameters:
{'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 10}

Best Cross-Validation Recall:
0.9193663424697908


### Random Forest Tuning Result

RandomizedSearchCV was used to tune the Random Forest classifier with recall as the scoring metric.

The best hyperparameters found were:

- n_estimators: 300
- max_depth: 10
- min_samples_split: 10
- min_samples_leaf: 2
- max_features: sqrt

The best cross-validation recall achieved was 0.9194.

Compared to the baseline cross-validation recall of 0.8702, the tuned Random Forest improved recall by approximately 4.92 percentage points. This improvement aligns with the project’s business objective of minimizing false negatives and identifying more customers likely to churn.

In [8]:
best_rf = rf_random_search.best_estimator_

In [9]:
y_pred_rf_tuned = best_rf.predict(X_test)

y_proba_rf_tuned = best_rf.predict_proba(X_test)[:, 1]

In [10]:
rf_tuned_accuracy = accuracy_score(y_test, y_pred_rf_tuned)
rf_tuned_precision = precision_score(y_test, y_pred_rf_tuned)
rf_tuned_recall = recall_score(y_test, y_pred_rf_tuned)
rf_tuned_f1 = f1_score(y_test, y_pred_rf_tuned)
rf_tuned_roc_auc = roc_auc_score(y_test, y_proba_rf_tuned)

In [11]:
print("=== Tuned Random Forest Performance ===\n")

print(f"Accuracy : {rf_tuned_accuracy:.4f}")
print(f"Precision: {rf_tuned_precision:.4f}")
print(f"Recall   : {rf_tuned_recall:.4f}")
print(f"F1 Score : {rf_tuned_f1:.4f}")
print(f"ROC-AUC  : {rf_tuned_roc_auc:.4f}")

=== Tuned Random Forest Performance ===

Accuracy : 0.9432
Precision: 0.7869
Recall   : 0.8862
F1 Score : 0.8336
ROC-AUC  : 0.9803


In [12]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_tuned))


Confusion Matrix:
[[1623   78]
 [  37  288]]


In [13]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.97      1701
           1       0.79      0.89      0.83       325

    accuracy                           0.94      2026
   macro avg       0.88      0.92      0.90      2026
weighted avg       0.95      0.94      0.94      2026



### Evaluation of Tuned Random Forest

After identifying the optimal hyperparameters using RandomizedSearchCV, the tuned Random Forest model was evaluated on the held-out test dataset.

The objective was to verify whether the improvements observed during cross-validation generalized to unseen data.

Particular emphasis was placed on recall, as the business objective prioritized minimizing false negatives and identifying customers at risk of churn.

## Tuned Random Forest Evaluation

The tuned Random Forest achieved a recall of 88.62%, improving upon the baseline model's recall of 85.23%.

Although accuracy, precision, F1 score, and ROC-AUC decreased slightly, the reduction in false negatives aligned more closely with the project's business objective.

The tuned model correctly identified 11 additional churners while reducing missed churners from 48 to 37. The increase in false positives was considered acceptable, as contacting additional customers is less costly than failing to identify customers likely to churn.

Therefore, the tuned Random Forest was selected as the preferred Random Forest configuration.


## 5.5 XGBoost Hyperparameter Tuning Strategy

RandomizedSearchCV was selected to tune the XGBoost classifier due to its computational efficiency when exploring large hyperparameter spaces.

The following hyperparameters were considered:

- n_estimators
- learning_rate
- max_depth
- subsample
- colsample_bytree
- min_child_weight

Stratified 5-fold cross-validation was used to maintain class balance across folds.

Recall was chosen as the optimization metric because the project prioritized identifying customers likely to churn and minimizing false negatives.

In [14]:
xgb_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "scale_pos_weight": [1, 3, 5]
}

In [15]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [16]:
xgb_random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring="recall",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [17]:
xgb_random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END colsample_bytree=0.6, learning_rate=0.05, max_depth=3, min_child_weight=7, n_estimators=100, scale_pos_weight=3, subsample=1.0; total time=   0.2s
[CV] END colsample_bytree=0.6, learning_rate=0.05, max_depth=3, min_child_weight=7, n_estimators=100, scale_pos_weight=3, subsample=1.0; total time=   0.2s
[CV] END colsample_bytree=0.6, learning_rate=0.05, max_depth=3, min_child_weight=7, n_estimators=100, scale_pos_weight=3, subsample=1.0; total time=   0.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, min_child_weight=3, n_estimators=300, scale_pos_weight=5, subsample=0.8; total time=   0.3s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, min_child_weight=3, n_estimators=300, scale_pos_weight=5, subsample=0.8; total time=   0.3s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, min_child_weight=3, n_estimators=300, scale_pos_weight=5, subsample=0.8; total time=   0.3s
[CV] 

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit

In [18]:
print("Best XGBoost Parameters:")
print(xgb_random_search.best_params_)

print("\nBest Cross-Validation Recall:")
print(xgb_random_search.best_score_)

Best XGBoost Parameters:
{'subsample': 0.8, 'scale_pos_weight': 5, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Best Cross-Validation Recall:
0.9470144414972002


In [19]:
best_xgb = xgb_random_search.best_estimator_

In [20]:
y_pred_xgb_tuned = best_xgb.predict(X_test)

y_proba_xgb_tuned = best_xgb.predict_proba(X_test)[:, 1]

In [21]:
xgb_tuned_accuracy = accuracy_score(y_test, y_pred_xgb_tuned)
xgb_tuned_precision = precision_score(y_test, y_pred_xgb_tuned)
xgb_tuned_recall = recall_score(y_test, y_pred_xgb_tuned)
xgb_tuned_f1 = f1_score(y_test, y_pred_xgb_tuned)
xgb_tuned_roc_auc = roc_auc_score(y_test, y_proba_xgb_tuned)

In [22]:
print("=== Tuned XGBoost Performance ===\n")

print(f"Accuracy : {xgb_tuned_accuracy:.4f}")
print(f"Precision: {xgb_tuned_precision:.4f}")
print(f"Recall   : {xgb_tuned_recall:.4f}")
print(f"F1 Score : {xgb_tuned_f1:.4f}")
print(f"ROC-AUC  : {xgb_tuned_roc_auc:.4f}")

=== Tuned XGBoost Performance ===

Accuracy : 0.9635
Precision: 0.8535
Recall   : 0.9323
F1 Score : 0.8912
ROC-AUC  : 0.9918


In [23]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_tuned))


Confusion Matrix:
[[1649   52]
 [  22  303]]


In [24]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_tuned))


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1701
           1       0.85      0.93      0.89       325

    accuracy                           0.96      2026
   macro avg       0.92      0.95      0.93      2026
weighted avg       0.97      0.96      0.96      2026



### Tuned XGBoost Evaluation

The tuned XGBoost model achieved the following test performance:

- Accuracy: 96.35%
- Precision: 85.35%
- Recall: 93.23%
- F1 Score: 89.12%
- ROC-AUC: 99.18%

Compared to the baseline XGBoost model, recall improved from 84.62% to 93.23%, reducing false negatives from 50 to 22.

The tuned model successfully identified 28 additional churners while maintaining high discriminative ability and strong overall predictive performance.

Given the project's business objective of minimizing missed churners, the tuned XGBoost model was selected as the best-performing model.

### Phase 5.6 

In [25]:
comparison_df = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Tuned Random Forest",
        "Baseline XGBoost",
        "Tuned XGBoost"
    ],
    "Accuracy": [
        0.9546,
        0.9432,
        0.9654,
        0.9635
    ],
    "Precision": [
        0.8629,
        0.7869,
        0.9322,
        0.8535
    ],
    "Recall": [
        0.8523,
        0.8862,
        0.8462,
        0.9323
    ],
    "F1 Score": [
        0.8576,
        0.8336,
        0.8871,
        0.8912
    ],
    "ROC-AUC": [
        0.9848,
        0.9803,
        0.9918,
        0.9918
    ]
})

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline Random Forest,0.9546,0.8629,0.8523,0.8576,0.9848
1,Tuned Random Forest,0.9432,0.7869,0.8862,0.8336,0.9803
2,Baseline XGBoost,0.9654,0.9322,0.8462,0.8871,0.9918
3,Tuned XGBoost,0.9635,0.8535,0.9323,0.8912,0.9918


## 5.6 Comparison of Models Before and After Tuning

The performance of the baseline and tuned versions of Random Forest and XGBoost was compared using Accuracy, Precision, Recall, F1 Score, and ROC-AUC.

Although tuning resulted in slight reductions in precision for both models, substantial improvements were observed in recall.

The tuned XGBoost model achieved the highest recall (93.23%) while maintaining strong overall predictive performance. Since the business objective prioritized minimizing false negatives and identifying customers likely to churn, the tuned XGBoost model was selected as the final candidate model.

## Final Model Selection After Hyperparameter Tuning

After comparing the baseline and tuned models, the tuned XGBoost model was selected as the final candidate model.

Although the baseline XGBoost model had slightly higher precision, the tuned XGBoost model achieved the highest recall.

Since the project’s primary business objective is to minimize missed churners, recall is the most important metric.

The tuned XGBoost model reduced false negatives from 50 to 22 compared to the baseline XGBoost model.

Therefore, the tuned XGBoost model best aligns with the business objective of identifying customers likely to churn.

In [26]:
import joblib
import os

os.makedirs("../src/models", exist_ok=True)

joblib.dump(best_rf, "../src/models/tuned_random_forest.pkl")
joblib.dump(best_xgb, "../src/models/tuned_xgboost.pkl")

print("Tuned Random Forest saved successfully.")
print("Tuned XGBoost saved successfully.")

Tuned Random Forest saved successfully.
Tuned XGBoost saved successfully.


In [27]:
print(os.path.exists("../src/models/tuned_random_forest.pkl"))
print(os.path.exists("../src/models/tuned_xgboost.pkl"))

True
True
